# ML Research pipeline
This notebook demonstrates ML pipeline using a public medical dataset. The goals are:

- present reproducible data processing and analysis
- compare classical and deep learning models
- optimize performance with hyperparameter search

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import ipywidgets as widgets

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# Load and explore the public biomedical dataset
raw = load_breast_cancer()
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df['target'] = raw.target

df['target_label'] = df['target'].map({0: 'malignant', 1: 'benign'})

print('Dataset shape:', df.shape)
print('Target classes:', raw.target_names)

display(df.head())

display(df.describe().T)


In [ ]:
# Data preprocessing and cleaning

print('Missing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

cleaned = df.drop_duplicates().reset_index(drop=True)

features = raw.feature_names
scaler = StandardScaler()
cleaned[features] = scaler.fit_transform(cleaned[features])

# Save metadata for research reporting
cleaned['target_label'] = cleaned['target'].map({0: 'malignant', 1: 'benign'})

display(cleaned.head())


In [ ]:
# Exploratory Data Analysis (EDA)

plt.figure(figsize=(8, 5))
sns.countplot(x='target_label', data=cleaned, palette='Set2')
plt.title('Malignancy label distribution')
plt.show()

plt.figure(figsize=(14, 12))
sns.heatmap(cleaned[features].corr(), cmap='vlag', center=0, square=True, linewidths=0.1)
plt.title('Feature correlation heatmap')
plt.show()

# PCA for visualizing the separation between classes
pca = PCA(n_components=2)
pca_projection = pca.fit_transform(cleaned[features])
plt.figure(figsize=(8, 6))
plt.scatter(pca_projection[:, 0], pca_projection[:, 1], c=cleaned['target'], cmap='coolwarm', alpha=0.8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection of breast cancer features')
plt.colorbar(label='Target')
plt.show()


In [ ]:
# Feature engineering
cleaned['radius_mean_to_texture_mean'] = cleaned['radius_mean'] / (cleaned['texture_mean'] + 1e-8)
cleaned['area_mean_to_perimeter_mean'] = cleaned['area_mean'] / (cleaned['perimeter_mean'] + 1e-8)

engineered_features = list(features) + ['radius_mean_to_texture_mean', 'area_mean_to_perimeter_mean']

print('New engineered features:')
print(engineered_features[-2:])

display(cleaned[engineered_features + ['target_label']].head())


In [ ]:
# Train-test split
X = cleaned[engineered_features]
y = cleaned['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)


In [ ]:
# Implement multiple models for computational biology classification
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'Trained {name}')


In [ ]:
# Model evaluation and comparison
metrics = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    metrics.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba) if proba is not None else None,
    })

metrics_df = pd.DataFrame(metrics).sort_values('f1', ascending=False)

display(metrics_df)

plt.figure(figsize=(10, 5))
plot_data = metrics_df.melt(id_vars='model', value_vars=['accuracy', 'precision', 'recall', 'f1'])
sns.barplot(data=plot_data, x='model', y='value', hue='variable')
plt.xticks(rotation=45)
plt.title('Model comparison across classification metrics')
plt.tight_layout()
plt.show()


In [ ]:
# Hyperparameter tuning in a research-style workflow
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, None],
    'learning_rate': [0.01, 0.1, 0.2],
}

gb = GradientBoostingClassifier(random_state=42)
grid = GridSearchCV(gb, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best parameters: ', grid.best_params_)
print('Best CV F1:', grid.best_score_)

best_gb = grid.best_estimator_
y_pred = best_gb.predict(X_test)
print('Test set classification report for tuned Gradient Boosting:')
print(classification_report(y_test, y_pred, target_names=raw.target_names, zero_division=0))


In [ ]:
# Interactive dashboard for model comparison
model_selector = widgets.Dropdown(options=list(trained_models.keys()), value='Random Forest', description='Model:')

output_area = widgets.Output()


def update_dashboard(model_name):
    with output_area:
        output_area.clear_output()
        model = trained_models[model_name]
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

        print(f'Selected model: {model_name}')
        print(classification_report(y_test, y_pred, target_names=raw.target_names, zero_division=0))

        cm = confusion_matrix(y_test, y_pred)
        fig = px.imshow(cm, text_auto=True, labels={'x': 'Predicted', 'y': 'Actual'}, x=raw.target_names, y=raw.target_names)
        fig.update_layout(title='Confusion matrix')
        fig.show()

        if proba is not None:
            roc = roc_auc_score(y_test, proba)
            print(f'ROC AUC: {roc:.3f}')

widgets.interact(update_dashboard, model_name=model_selector)

display(output_area)


In [ ]:
# Predictions, feature insights, and research conclusions
best_model_name = metrics_df.loc[metrics_df['f1'].idxmax(), 'model']
best_model = trained_models[best_model_name]
print(f'Best performing model by F1: {best_model_name}')

predictions = best_model.predict(X_test)
result_df = X_test.copy()
result_df['actual'] = y_test.values
result_df['predicted'] = predictions
result_df['correct'] = result_df['actual'] == result_df['predicted']

print('Test accuracy:', accuracy_score(y_test, predictions))
display(result_df.head())

if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=X_test.columns).sort_values(ascending=False)
    fig = px.bar(importance.head(12), title='Top 12 Feature Importances')
    fig.show()
else:
    print('Feature importances are not available for this model.')
